# Unidad 1: Del origen a Spark — Contexto histórico y tecnológico

**Tecnicatura en Datos | Procesamiento con Apache Spark**  
Unidad 1 de 10 | Duración estimada: 2:30 hs

> **Nota Databricks:** En este notebook los bloques de código ilustran cómo los conceptos históricos se traducen a Spark moderno. El objeto `spark` ya está disponible en el entorno Databricks.

---
## 1. El problema que lo inició todo

A principios de los 2000, **Google** enfrentaba un desafío sin precedentes: indexar cientos de terabytes de páginas web con hardware común y económico.

La solución convencional (**scale-up**) tenía límites físicos y económicos claros. Google eligió el paradigma del **scale-out**:

| Paradigma | Estrategia | Limitación |
|---|---|---|
| Scale-up | Máquinas más potentes | Límite físico y costo exponencial |
| Scale-out | Muchas máquinas baratas | Requiere software distribuido |

Esta decisión obligó a resolver nuevos problemas:
- ¿Cómo distribuir datos en miles de discos?
- ¿Qué pasa cuando una máquina falla?
- ¿Cómo coordinar trabajo entre miles de nodos?

---
## 2. Google File System (GFS) — 2003

GFS es un **sistema de archivos distribuido** diseñado para:
- Hardware commodity (máquinas comunes y baratas)
- Archivos muy grandes (decenas o cientos de GB)
- Entornos donde **las fallas son la norma**, no la excepción

### Arquitectura de GFS

```
┌─────────────┐
│  Master Node │  ← metadata (nombres, ubicaciones, permisos)
└──────┬───────┘
       │ coordina
┌──────▼──────────────────────────────────┐
│  ChunkServer 1 │ ChunkServer 2 │ ... N  │  ← datos reales
└─────────────────────────────────────────┘
```

**Mecanismo clave: replicación**  
Cada chunk (fragmento de 64 MB) se replica en **3 nodos** distintos. Si uno falla, los datos no se pierden.

### Legado
GFS inspiró directamente **HDFS** (Hadoop Distributed File System), la implementación open source que usa Spark hasta hoy.

---
## 3. MapReduce — 2004

GFS resolvió el **almacenamiento**. MapReduce resolvió el **procesamiento**.

El modelo divide cualquier problema en dos fases:

| Fase | Qué hace | Paralelismo |
|---|---|---|
| **Map** | Transforma cada elemento en pares clave-valor | Total: sin comunicación entre workers |
| **Shuffle** | Agrupa pares por clave (automático) | El framework lo maneja |
| **Reduce** | Combina valores por clave | Por clave |

### Ejemplo clásico: Word Count

```
Entrada:  "el gato y el perro"

Map:      (el,1) (gato,1) (y,1) (el,1) (perro,1)

Shuffle:  el→[1,1]  gato→[1]  y→[1]  perro→[1]

Reduce:   el→2  gato→1  y→1  perro→1
```

---
## 4. Limitaciones de MapReduce

A pesar de su éxito, MapReduce tenía problemas fundamentales:

### 4.1 Escritura en disco en cada paso

Cada fase escribe en disco. Para algoritmos iterativos (como ML), esto es muy costoso:

```
Iteración 1: leer HDFS → procesar → escribir HDFS
Iteración 2: leer HDFS → procesar → escribir HDFS
Iteración N: leer HDFS → procesar → escribir HDFS
```

### 4.2 Modelo rígido de dos fases
Fuerza a expresar todo como Map + Reduce. Algoritmos complejos requieren múltiples jobs encadenados.

### 4.3 Alta latencia
Incluso jobs simples tienen latencias de varios minutos. No apto para trabajo interactivo.

---
## 5. Apache Hadoop — 2006

Doug Cutting implementó GFS y MapReduce como software **open source** en Java, donado a la Apache Software Foundation.

**Componentes principales:**
- **HDFS**: implementación open source de GFS
- **MapReduce**: implementación open source del modelo de Google
- **YARN** (v2): gestor de recursos del clúster

Hadoop democratizó el procesamiento distribuido, pero heredó las limitaciones de MapReduce.

---
## 6. El nacimiento de Apache Spark — 2009

Investigadores de UC Berkeley (AMPLab) desarrollaron Spark para resolver los problemas de MapReduce:

| Problema MapReduce | Solución Spark |
|---|---|
| Escritura en disco en cada paso | Procesamiento **en memoria (RAM)** |
| Modelo Map + Reduce rígido | DAG flexible con múltiples transformaciones |
| Alta latencia | Modo interactivo y batch en el mismo motor |
| Solo batch | Streaming, ML, SQL en un único framework |

### La ventaja de la memoria

```
MapReduce (disco):   Iteración = leer disco + procesar + escribir disco
Spark (memoria):     Iteración = procesar en RAM (sin I/O)

Resultado: Spark es ~100x más rápido en cargas iterativas
```

---
## 7. Demostración: Spark vs MapReduce conceptual

El siguiente código muestra cómo Spark resuelve el Word Count que MapReduce hacía con decenas de líneas de Java:

In [ ]:
# En Databricks: spark y sc ya están disponibles
# Para entorno local, descomentar:
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("Unidad1").getOrCreate()

sc = spark.sparkContext

# Word Count equivalente al ejemplo MapReduce — en 4 líneas
texto = sc.parallelize([
    "el gato y el perro",
    "el gato duerme",
    "el perro juega",
])

conteo = (
    texto
    .flatMap(lambda linea: linea.split(" "))  # fase MAP
    .map(lambda palabra: (palabra, 1))         # fase MAP
    .reduceByKey(lambda a, b: a + b)           # fase REDUCE
    .sortBy(lambda par: par[1], ascending=False)
)

print("Resultado Word Count:")
for palabra, freq in conteo.collect():
    print(f"  '{palabra}': {freq}")

**Resultado esperado:**
```
Resultado Word Count:
  'el': 4
  'gato': 2
  'perro': 2
  'y': 1
  'duerme': 1
  'juega': 1
```

> Lo que en MapReduce (Java) requería ~70 líneas de código boilerplate, en Spark se expresa en 4 líneas.

---
## 8. El ecosistema Spark actual

Spark se convirtió en un ecosistema completo:

In [ ]:
# Ver la versión de Spark en uso
print(f"Versión de Spark: {spark.version}")
print(f"Versión de Python: {spark.sparkContext.pythonVer}")
print(f"Master: {spark.sparkContext.master}")
print(f"App Name: {spark.sparkContext.appName}")

---
## 9. Línea de tiempo resumen

```
2003 ──► Google publica paper de GFS
2004 ──► Google publica paper de MapReduce
2006 ──► Apache Hadoop (open source de GFS + MapReduce)
2009 ──► Nace Apache Spark (UC Berkeley AMPLab)
2013 ──► Spark donado a Apache Software Foundation
2014 ──► Spark 1.0 — DataFrames y Spark SQL
2016 ──► Spark 2.0 — SparkSession unificada
2020 ──► Spark 3.0 — Adaptive Query Execution
2024 ──► Spark 4.0 — mejoras en Spark Connect y Python
```

---
## 10. Preguntas de revisión

1. ¿Por qué Google eligió scale-out en lugar de scale-up?
2. ¿Cuál es la diferencia fundamental entre `map` y `reduce`?
3. ¿Por qué MapReduce es lento para algoritmos iterativos?
4. ¿Qué ventaja clave tiene Spark sobre MapReduce?
5. ¿Qué es HDFS y de qué sistema se inspiró?

---
**Próxima unidad:** Arquitectura de Apache Spark